# Day 20 · 動手接上遠端 Agent：A2A Exposing 與 Consuming

> 第三部・戰術編排　|　🔀 需要兩個程序（notebook 自動起停）

**前置需求**：🔑 需要 Gemini API 金鑰　📦 需要 `google-adk[a2a]`

**對應文章**：`Day 20 - 動手接上遠端 Agent：A2A Exposing 與 Consuming.md`

## 今天要學會

1. 用 `to_a2a()` 把 agent 變成服務
2. 用 `RemoteA2aAgent` 消費遠端 agent
3. 讀懂 agent card

> 本日 notebook 會自己起一個 uvicorn 子程序當「遠端」，
> 跑完最後一格會自動關掉。**請從頭依序執行到最後一格。**

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

import httpx
import uvicorn

print("httpx", httpx.__version__, "| uvicorn", uvicorn.__version__)

google-adk 2.8.0
httpx 0.28.1 | uvicorn 0.46.0


## 1. 兩個對稱的動作

A2A 在 ADK 這邊只有兩個 API，剛好對稱：

| 你想做的事 | 用什麼 | 產出 |
|---|---|---|
| **Exposing**：把自己的 agent 開出去 | `to_a2a(agent)` | 一個 Starlette app |
| **Consuming**：接上別人的 agent | `RemoteA2aAgent(name, agent_card=...)` | 一個可以當 sub_agent 用的 agent |

關鍵在最後一欄：**`RemoteA2aAgent` 拿到之後跟本地 agent 用起來一模一樣**，
可以直接丟進 `sub_agents`。主 agent 根本不知道它在跟別台機器講話。

## 2. Exposing：先寫一支要被開出去的 agent

`to_a2a()` 吃一個 agent，回傳 Starlette app。真正跑起來要靠 uvicorn，
所以我們把它寫成獨立的 `.py`，等一下用子程序啟動。

In [2]:
import shutil
import subprocess
import sys
import time
from pathlib import Path

PORT = 8931
HOST = "localhost"                      # ⚠️ 這個字串等一下會很重要
WORK = Path.cwd() / "_day20"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

SERVER = WORK / "inventory_server.py"
SERVER.write_text(f'''"""被開出去的遠端 agent。"""
import warnings

warnings.filterwarnings("ignore")

import sys

sys.path.insert(0, {str(PROJECT_ROOT)!r})

from shared import get_model, quiet

quiet()

from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.adk.agents import LlmAgent

STOCK = {{"A-100": 42, "B-200": 0}}


def check_stock(sku: str) -> dict:
    """查詢商品庫存數量。

    Args:
        sku: 商品編號，例如 A-100。
    """
    return {{"sku": sku, "quantity": STOCK.get(sku, 0)}}


root_agent = LlmAgent(
    name="inventory_agent",
    model=get_model(),
    description="查詢商品庫存與補貨狀態",
    instruction="你是庫存助理。一律呼叫 check_stock 查詢，用繁體中文一句話回答。",
    tools=[check_stock],
)

# ⭐ 這一行就是 Exposing 的全部
app = to_a2a(root_agent, host={HOST!r}, port={PORT})

if __name__ == "__main__":
    import uvicorn

    uvicorn.run(app, host={HOST!r}, port={PORT}, log_level="error")
''')

print(f"已寫出 {SERVER.name}（{SERVER.stat().st_size} bytes）")
print("\n⭐ Exposing 的關鍵只有一行：")
print("     app = to_a2a(root_agent, host=..., port=...)")

已寫出 inventory_server.py（1003 bytes）

⭐ Exposing 的關鍵只有一行：
     app = to_a2a(root_agent, host=..., port=...)


## 3. 把它跑起來（子程序）

`to_a2a` 只給你 app，**啟動是你的事**。這裡用 `subprocess.Popen` 起 uvicorn，
然後輪詢 agent card 端點，確認真的活了才往下走。

In [3]:
LOG = WORK / "server.log"
proc = subprocess.Popen(
    [sys.executable, str(SERVER)],
    stdout=LOG.open("w"), stderr=subprocess.STDOUT,
)

CARD_URL = f"http://{HOST}:{PORT}/.well-known/agent-card.json"

ready = False
for _ in range(60):
    if proc.poll() is not None:          # 子程序自己死了
        break
    try:
        if httpx.get(CARD_URL, timeout=2).status_code == 200:
            ready = True
            break
    except Exception:
        time.sleep(0.5)

if ready:
    print(f"✅ 遠端 agent 已在 {HOST}:{PORT} 上線")
else:
    print("❌ 起不來，server log：")
    print(LOG.read_text()[-800:])

✅ 遠端 agent 已在 localhost:8931 上線


## 4. 讀懂 Agent Card

文章只給了 agent card 的示意圖。這裡把**真的那一份**抓下來看。

重點：**你沒有手寫這張卡，是 `to_a2a()` 從 agent 自動產生的。**

In [4]:
import json

card = httpx.get(CARD_URL, timeout=5).json()
print(json.dumps(card, ensure_ascii=False, indent=2))

{
  "name": "inventory_agent",
  "description": "查詢商品庫存與補貨狀態",
  "supportedInterfaces": [
    {
      "url": "http://localhost:8931",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "0.0.1",
  "capabilities": {
    "streaming": false,
    "pushNotifications": false
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "skills": [
    {
      "id": "inventory_agent",
      "name": "model",
      "description": "查詢商品庫存與補貨狀態",
      "tags": [
        "llm"
      ]
    },
    {
      "id": "inventory_agent-check_stock",
      "name": "check_stock",
      "description": "查詢商品庫存數量。\n\nArgs:\n    sku: 商品編號，例如 A-100。",
      "tags": [
        "llm",
        "tools"
      ]
    }
  ]
}


### 這張卡怎麼來的

逐欄對照，你會發現每一格都能追到 agent 的某個屬性：

| 卡片欄位 | 來自 |
|---|---|
| `name` | `LlmAgent(name=...)` |
| `description` | `LlmAgent(description=...)` |
| `supportedInterfaces[].url` | `to_a2a(host=, port=)` |
| `skills[0]` | agent 本體（tag 是 `llm`） |
| `skills[1:]` | **每一個 tool 各生一個 skill**（tag 多了 `tools`） |
| `capabilities.streaming` | 由 ADK 決定 |

In [5]:
print("這張卡宣告的連線位址:")
for itf in card.get("supportedInterfaces", []):
    print(f"  • {itf['url']}  ({itf.get('protocolBinding')})")

print("\nskills（注意工具也變成 skill 了）:")
for s in card.get("skills", []):
    print(f"  • id={s['id']:<32} tags={s.get('tags')}")

print("\n→ 所以 ⚠️ **工具的 docstring 會被公開出去**。")
print("   內部欄位名稱、商業邏輯，寫在 docstring 裡就等於對外揭露。")

這張卡宣告的連線位址:
  • http://localhost:8931  (JSONRPC)

skills（注意工具也變成 skill 了）:
  • id=inventory_agent                  tags=['llm']
  • id=inventory_agent-check_stock      tags=['llm', 'tools']

→ 所以 ⚠️ **工具的 docstring 會被公開出去**。
   內部欄位名稱、商業邏輯，寫在 docstring 裡就等於對外揭露。


## 5. Consuming：接上去

`RemoteA2aAgent` 只要一個名字和一張卡的網址。

In [6]:
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from google.adk.runners import InMemoryRunner


def make_remote():
    """每次都給新實例——agent 只能有一個父節點（Day 16）。"""
    return RemoteA2aAgent(
        name="remote_inventory",
        agent_card=CARD_URL,
        description="遠端庫存 agent，可查詢商品庫存數量",
    )


r_direct = InMemoryRunner(agent=make_remote(), app_name="day20")
sid = await new_session(r_direct)
print("問：A-100 還有幾個？")
print("答：", await ask(r_direct, "A-100 還有幾個？", session_id=sid))

問：A-100 還有幾個？


答： A-100 目前庫存還有 42 個。


## 6. ⚠️ 本日最重要的坑：origin 必須完全一致

這個坑會讓你 debug 很久，因為**它是靜默失敗的**。

Agent card 裡宣告的 URL 是 `http://localhost:8931`。
如果你改用 `http://127.0.0.1:8931` 去抓卡——**同一台機器、同一個埠**——
A2A client 會拒絕連線，因為它要求「卡片宣告的 RPC URL」與「你抓卡的位置」同 origin。

而且它**不會拋例外**，只會回一個空的回應。

In [7]:
BAD_URL = CARD_URL.replace("localhost", "127.0.0.1")
print("改用這個網址抓卡:", BAD_URL)
print("（卡片本身抓得到嗎？", httpx.get(BAD_URL, timeout=5).status_code == 200, "）\n")

from google.genai import types

bad_remote = RemoteA2aAgent(
    name="bad_remote", agent_card=BAD_URL, description="遠端庫存 agent",
)
r_bad = InMemoryRunner(agent=bad_remote, app_name="day20")
sid_bad = await new_session(r_bad)

# ⚠️ 錯誤只在「第一次」初始化時報一次，所以要直接看事件、不能先呼叫 ask()
msg = types.Content(role="user", parts=[types.Part(text="A-100 還有幾個？")])
texts, errors = [], []
async for ev in r_bad.run_async(user_id="student", session_id=sid_bad, new_message=msg):
    if ev.content and ev.content.parts:
        texts += [p.text for p in ev.content.parts if p.text]
    if ev.error_message:
        errors.append(ev.error_message)

print(f"收到的文字內容: {texts}   ← 空的，而且沒有拋例外")
print("\n❌ 真正的錯誤藏在 event.error_message：")
for e in errors:
    print("  ", e)

改用這個網址抓卡: http://127.0.0.1:8931/.well-known/agent-card.json
（卡片本身抓得到嗎？ True ）

收到的文字內容: []   ← 空的，而且沒有拋例外

❌ 真正的錯誤藏在 event.error_message：
   Failed to initialize remote A2A agent: Failed to initialize remote A2A agent bad_remote: Agent card RPC URL must have the same origin as the location the card was fetched from (http://127.0.0.1:8931/.well-known/agent-card.json): http://localhost:8931


**記住這條規則**：`127.0.0.1` 跟 `localhost` 對 A2A 來說是**不同的 origin**。

所以 `to_a2a(host=...)` 填什麼，client 就必須用什麼去抓卡。
部署到正式環境時，`host` 要填**對外的網域名稱**，不是 `0.0.0.0`。

> 排查順序：`ask()` 回空字串 → 去看 `event.error_message` → 十次有九次是 origin 或網址打錯。

## 7. 混合委派：本地 + 遠端放在同一組 `sub_agents`

這是 A2A 真正好用的地方——**主 agent 分不出誰在本機、誰在遠端。**

In [8]:
from google.adk.agents import LlmAgent

local_helper = LlmAgent(
    name="local_helper", model=get_model(),
    description="回答一般常識問題，不涉及庫存",
    instruction="用一句話回答，繁體中文。",
)

boss = LlmAgent(
    name="boss", model=get_model(),
    instruction="庫存問題轉給 remote_inventory，其他問題轉給 local_helper。不要自己回答。",
    sub_agents=[make_remote(), local_helper],      # ← 一個遠端、一個本地
)

r_mix = InMemoryRunner(agent=boss, app_name="day20")
for q in ["B-200 還有貨嗎？", "台灣最高的山是哪一座？"]:
    sid_m = await new_session(r_mix)
    print(f"📥 {q}")
    await ask(r_mix, q, session_id=sid_m, trace=True)
    print()

📥 B-200 還有貨嗎？


  🔧 [boss] 呼叫 transfer_to_agent({'agent_name': 'remote_inventory'})
  ↩️  [boss] transfer_to_agent 回傳 {'result': None}


  💬 [remote_inventory] 商品 B-200 目前已經沒有庫存。

📥 台灣最高的山是哪一座？


  🔧 [boss] 呼叫 transfer_to_agent({'agent_name': 'local_helper'})
  ↩️  [boss] transfer_to_agent 回傳 {'result': None}


  💬 [local_helper] 台灣最高的山是玉山。



看事件串流：兩次都是 `transfer_to_agent`，
**格式完全一樣**——一個轉給了跨程序的 HTTP 服務，一個轉給了同程序的物件。

這就是 Day 19 說的「對主 agent 來說，遠端 agent 跟本地 agent 長得一模一樣」。

## 8. 收工：關掉子程序

⚠️ 忘了關，那個埠會一直被佔著，下次重跑 notebook 會 `address already in use`。

In [9]:
proc.terminate()
try:
    proc.wait(timeout=10)
except subprocess.TimeoutExpired:
    proc.kill()
    proc.wait(timeout=5)

still_up = True
try:
    httpx.get(CARD_URL, timeout=2)
except Exception:
    still_up = False

shutil.rmtree(WORK, ignore_errors=True)
print("子程序已結束，回傳碼:", proc.returncode)
print("埠還開著嗎？", still_up)
print("工作目錄已清理:", not WORK.exists())

子程序已結束，回傳碼: -15
埠還開著嗎？ False
工作目錄已清理: True


## 9. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| `ask()` 回空字串、也沒報錯 | **去看 `event.error_message`**，多半是 origin 或網址錯 |
| `Agent card RPC URL must have the same origin...` | `127.0.0.1` ≠ `localhost`，兩邊要用同一個字串 |
| `address already in use` | 上次的子程序沒關掉，先 `terminate()` |
| 部署後遠端連不上 | `to_a2a(host=...)` 填了 `0.0.0.0`，卡片就會宣告連不到的位址 |
| 不想公開的細節出現在卡片上 | **工具 docstring 會變成 skill description**，卡片是公開的 |
| `ModuleNotFoundError: No module named 'a2a'` | 沒裝 extra，要 `google-adk[a2a]` |
| server 起不來但 notebook 沒說 | 看 `_day20/server.log`，子程序的錯誤不會出現在 notebook |

## 10. 動手練習

1. 把 server 的 `check_stock` 再加一個工具（例如 `restock_eta`），
   重啟後重抓卡片，確認 `skills` 多了一個、而 client 端**完全不用改**。
2. 把 `to_a2a(host=...)` 改成 `0.0.0.0` 再跑一次，
   看卡片宣告的位址變成什麼、client 為什麼就連不上了。
3. 把 `boss` 的 `remote_inventory` 換成 `mode="single_turn"`（Day 18），
   確認控制權會回到 boss 手上再收尾。
4. 故意把 server 關掉之後才呼叫 `remote`，
   觀察錯誤跟第 6 節的 origin 錯誤長得像不像——這會決定你以後怎麼排查。

## 本日回顧

- **Exposing 只有一行**：`app = to_a2a(root_agent, host=, port=)`；啟動要自己用 uvicorn。
- **Consuming 也只有一行**：`RemoteA2aAgent(name=..., agent_card=<卡片網址>)`。
- **Agent card 是自動產生的**，`name` / `description` / **每一個 tool** 都會變成卡片內容。
  ⚠️ 所以工具的 docstring 等於對外文件。
- ⚠️ **`127.0.0.1` 與 `localhost` 是不同 origin**，不一致就連不上，
  而且**靜默失敗**——`ask()` 回空字串，真正的錯誤在 `event.error_message`。
- **遠端 agent 可以直接放進 `sub_agents`**，跟本地 agent 混用，事件串流長得一模一樣。
- 起了子程序就要記得 `terminate()`，否則埠會一直被佔著。

---
**下一天 → `../day21_live_and_voice/`**